# Class 1: Understanding Exposure

## Objective
In this class, you will learn what **exposure** means in vulnerability assessment and use GIS analysis to determine which properties (parcels) are exposed to the 100-year floodplain in our study area.

By the end of this notebook, you will:
1. Understand what exposure means (is the asset in the path of the hazard?)
2. Learn about the 100-year floodplain and why it's used for disaster planning
3. Perform a **spatial join** to match parcels to flood zones
4. Create exposure scores: 1 = exposed, 0 = not exposed
5. Classify assets by type (residential, commercial, industrial, etc.)
6. Visualize and summarize the results on maps and in tables

**Important**: You do not need to be a programmer to use this notebook. We'll walk through each step together.

## Step 1: Mount Google Drive and Install Libraries

First, we'll connect to your Google Drive and install the Python libraries we need for mapping and data analysis.

**What's happening here:**
- Google Colab is a free programming environment, but it's separate from your computer
- We need to connect it to your Google Drive to access your data files
- We'll install `geopandas`, `folium`, and other tools for geographic data analysis

In [ ]:
# Mount Google Drive so we can access our data files
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!")

In [ ]:
# Install and load our mapping and data tools
# (Run this cell every time you open this notebook)
# ============================================================
!pip install geopandas fiona shapely pyproj requests folium seaborn contextily --quiet

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
import os
import json
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries loaded successfully!")
print("\nLibraries installed:")
print("  - geopandas: for geographic data")
print("  - pandas: for tables and data")
print("  - numpy: for numbers and calculations")
print("  - matplotlib: for making charts and maps")
print("  - folium: for interactive web maps")

## Step 2: Set Up Your Data Directory

We'll define the path to your study area data. All your data files are stored in a **GeoPackage** (a single file that contains multiple map layers).

In [ ]:
# Set the base directory where our data is stored
BASE_DIR = '/content/drive/MyDrive/MSER_510_VULNERABILTY'
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Path to our GeoPackage file (contains parcels, flood zones, etc.)
GPKG_PATH = os.path.join(DATA_DIR, 'vulnerability_risk_data.gpkg')
if not os.path.exists(GPKG_PATH):
    _alt = os.path.join(BASE_DIR, 'vulnerability_risk_data.gpkg')
    if os.path.exists(_alt):
        GPKG_PATH = _alt
        print(f"  Note: Using base data from: {GPKG_PATH}")
CLASS0_GPKG = GPKG_PATH  # Alias for consistency with other notebooks

# Verify the file exists
if os.path.exists(GPKG_PATH):
    print(f"✓ Found data file: {GPKG_PATH}")
    print(f"  File size: {os.path.getsize(GPKG_PATH) / 1024 / 1024:.1f} MB")
else:
    print(f"✗ Data file not found at: {GPKG_PATH}")
    print(f"  Please check that the file exists in your Google Drive")

## What is Exposure?

In disaster risk management, **exposure** answers this question:

> **"Is this asset (building, parcel, infrastructure) in the path of the hazard?"**

Think of it this way:
- **Hazard** = The dangerous event (a flood, earthquake, hurricane, etc.)
- **Asset** = Something we want to protect (homes, businesses, roads, parks)
- **Exposure** = Does the asset sit in the area where the hazard could occur?

### Exposure is Binary (Yes/No)
For this class, we're using simple **binary scoring**:
- **1** = The parcel is exposed to the 100-year floodplain (flood could happen here)
- **0** = The parcel is NOT exposed (located outside the floodplain)

This is different from vulnerability or risk:
- **Exposure**: Is it in the hazard zone? (yes/no)
- **Vulnerability**: If hit by the hazard, how much damage would it suffer? (depends on building type, construction, etc.)
- **Risk**: Exposure × Vulnerability × Probability (the actual danger level)

We focus on exposure first because you can't reduce risk if the asset isn't even exposed!

## The 100-Year Floodplain: Why It Matters

The **100-year floodplain** is the area that has a **1% chance of flooding in any given year**.

### Key Misconception
❌ **"100-year floodplain" does NOT mean "floods once every 100 years"**

✓ **It means "has a 1% annual probability"** — so it could flood multiple years in a row, or go 200+ years without flooding.

### Why Use the 100-Year Standard?
1. **Regulatory standard** — FEMA and government agencies use this for flood insurance and building codes
2. **Risk management balance** — It protects against serious floods without being overly restrictive
3. **Insurance requirement** — Banks require flood insurance for properties in the 100-year floodplain
4. **Zoning and development** — Many areas restrict building in these zones

For comparison:
- 500-year floodplain = 0.2% annual probability (less frequent, larger area)
- 10-year floodplain = 10% annual probability (more frequent, smaller area)

Today, we focus on **100-year flood exposure** because it's the regulatory standard used for disaster planning and building codes.

## Step 3: Load the Parcels and Flood Zone Data

We'll load two layers from the GeoPackage:
1. **Parcels** — individual property boundaries in our study area
2. **Flood zones** — areas that are at risk from flooding

Both will be loaded as **GeoDataFrames** (like Excel spreadsheets, but with geography information).

In [ ]:
# Load parcels from the GeoPackage
try:
    parcels = gpd.read_file(GPKG_PATH, layer='parcels')
    print(f"✓ Loaded {len(parcels):,} parcels")
    print(f"  Coordinate system: {parcels.crs}")
except Exception as e:
    print(f"✗ Could not load parcels layer: {e}")
    parcels = None

# Load flood zones from the GeoPackage
try:
    flood_zones = gpd.read_file(GPKG_PATH, layer='flood_zones')
    print(f"✓ Loaded {len(flood_zones):,} flood zone features")
    print(f"  Coordinate system: {flood_zones.crs}")
except Exception as e:
    print(f"✗ Could not load flood_zones layer: {e}")
    flood_zones = None

## Step 3b: Explore Land Use Classification Fields

Before we analyze exposure, we need to choose how to classify our parcels. The NC OneMap parcel data includes several fields that describe what a parcel is used for. Different fields offer different levels of detail.

We'll compare **three classification fields** side by side so you can pick the one that best fits your analysis:

| Field | Description |
|-------|-------------|
| `parusedesc` | Primary land use description (e.g., RESIDENTIAL IMPROVED, COMMERCIAL IMPROVED) |
| `parusedsc2` | Secondary land use description (may have finer categories) |
| `parvaltype` | Parcel value type classification |

Review all three tables below, then in the next cell you'll choose which field to use and pick your two **community assets** from that field.

In [ ]:
# ============================================================
# SUMMARY TABLES: Compare three land use classification fields
# ============================================================
# Review all three tables to decide which field works best
# for your community asset analysis.
# ============================================================

candidate_fields = ['parusedesc', 'parusedsc2', 'parvaltype']

for field in candidate_fields:
    print("\n" + "=" * 70)
    if field in parcels.columns:
        values = parcels[field].fillna('(blank/missing)')
        summary = values.value_counts().reset_index()
        summary.columns = [field, 'Count']
        summary['Pct'] = (summary['Count'] / len(parcels) * 100).round(1)
        print(f"  FIELD: {field}  —  {values.nunique()} unique values")
        print(f"  Blanks/missing: {(parcels[field].isna() | (parcels[field] == '')).sum():,}")
        print("=" * 70)
        # Display the full table
        for _, row in summary.iterrows():
            print(f"    {row[field]:40s}  {int(row['Count']):>6,}  ({row['Pct']:>5.1f}%)")
    else:
        print(f"  FIELD: {field}  —  NOT FOUND in parcel data")
        print("=" * 70)
        print(f"    This field was not downloaded from NC OneMap.")
        print(f"    It may not be available for Buncombe County.")

print("\n" + "=" * 70)
print("  Review the tables above, then choose your field and assets below.")
print("=" * 70)

### Review and Update Your Community Asset Selections

In Class 0 you chose your community asset groups as lists of **parusecode** values. Those selections are saved in `course_config.json`.

This is the **last notebook** where you can change your selections. If you want to update them, uncomment and edit the lists in the next cell and re-run it. Your changes will be saved to `course_config.json` and used in all subsequent classes (2-8).

If you're happy with your current selections, just run the next cell to load them.

In [ ]:
# ============================================================
# COMMUNITY ASSET SELECTIONS (parusecode lists)
# ============================================================
# Your community asset selections are stored in course_config.json,
# which was created in Class 0. This cell loads them from that file.
#
# You can edit the lists below and re-run this cell to change
# your selections. Changes are saved back to course_config.json
# and will carry forward to Class 2 and beyond.
#
# This is the LAST notebook where you can change these selections.
# Classes 2-8 will load whatever is saved here.
# ============================================================

# Fallback defaults — only used if course_config.json does not exist
# (i.e., you haven't run Class 0 yet)
default_asset_1 = ['100', '101', '105', '120', '121', '170', '173', '411', '416']
default_asset_2 = ['340', '341', '365', '405', '414', '415', '417', '421', '423',
                   '425', '426', '430', '431', '432', '434', '435', '438', '440',
                   '444', '446', '447', '448', '450', '454', '455', '456', '462',
                   '464', '466', '468', '470', '471', '472', '476', '477', '478',
                   '480', '481', '483', '490', '492', '494', '495', '512', '541',
                   '543', '544', '551', '554']

# Load from course_config.json (created in Class 0)
config_path = os.path.join(DATA_DIR, 'course_config.json')
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        course_config = json.load(f)
    community_asset_1 = course_config.get('community_asset_1', default_asset_1)
    community_asset_2 = course_config.get('community_asset_2', default_asset_2)
    print(f"  Loaded from config: {config_path}")
else:
    community_asset_1 = default_asset_1
    community_asset_2 = default_asset_2
    print(f"  Config not found — using defaults. Run Class 0 first to set your selections.")

# ============================================================
# TO CHANGE: Uncomment and edit these lists, then re-run this cell.
# Your changes will be saved to course_config.json and used in
# all subsequent classes (2-8).
# ============================================================
# community_asset_1 = ['100', '101', '105', '120', '121', '170', '173', '411', '416']
# community_asset_2 = ['340', '341', '365', ...]
# ============================================================

# Validate and display
if 'parusecode' in parcels.columns:
    available = parcels['parusecode'].dropna().astype(str).unique().tolist()

    community_assets_all = community_asset_1 + community_asset_2
    for i, asset_codes in enumerate([community_asset_1, community_asset_2], 1):
        print(f"\n  Community Asset {i}: {asset_codes}")
        total = 0
        for code in asset_codes:
            code_str = str(code)
            if code_str in available:
                count = (parcels['parusecode'].astype(str) == code_str).sum()
                desc_match = parcels.loc[parcels['parusecode'].astype(str) == code_str, 'parusedesc']
                desc = desc_match.iloc[0] if len(desc_match) > 0 else '(no description)'
                print(f"    parusecode '{code_str}' — {desc} — {count:,} parcels")
                total += count
            else:
                print(f"    WARNING: parusecode '{code_str}' NOT FOUND in data!")
        print(f"    TOTAL for Asset {i}: {total:,} parcels")

    # Save updated config
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            course_config = json.load(f)
    else:
        course_config = {}

    course_config['community_asset_1'] = [str(c) for c in community_asset_1]
    course_config['community_asset_2'] = [str(c) for c in community_asset_2]

    with open(config_path, 'w') as f:
        json.dump(course_config, f, indent=2)
    print(f"\n  Config saved to: {config_path}")
else:
    print("  ERROR: parusecode field not found in parcels data.")

## Step 4: Filter Flood Zones for Exposure Analysis

The flood_zones layer contains multiple flood categories (Floodway, 100-year, 500-year, etc.).

We'll **filter** the data to keep the **Floodway** and **100-year floodplain** — both are part of the regulatory flood hazard area used for exposure analysis.

- **Floodway:** The channel of a river plus adjacent areas that must be kept free of obstruction so flood waters can pass. This is the *most dangerous* part of the floodplain.
- **100-year (1% annual chance):** The broader area around the floodway with a 1% chance of flooding in any given year.

Together, these form the **Special Flood Hazard Area (SFHA)** — the regulatory standard for flood insurance and building codes.

**What "filter" means:** We're creating a new table with only the rows that match our criteria (flood_category is '100-year' OR 'Floodway').

In [ ]:
# Filter flood zones to keep Floodway + 100-year floodplain
# Both are part of the Special Flood Hazard Area (SFHA) used for exposure
exposure_categories = ['100-year', 'Floodway']
flood_zones_100yr = flood_zones[flood_zones['flood_category'].isin(exposure_categories)].copy()

print(f"✓ Filtered to Floodway + 100-year floodplain")
print(f"  Original flood zones: {len(flood_zones):,}")
print(f"  Exposure flood zones: {len(flood_zones_100yr):,}")
if 'flood_category' in flood_zones_100yr.columns:
    for cat, count in flood_zones_100yr['flood_category'].value_counts().items():
        print(f"    {cat}: {count:,} features")
print(f"  Total area: {flood_zones_100yr.geometry.area.sum() / 1_000_000:.1f} million square meters")

## Step 5: Spatial Join — Match Parcels to Flood Zones

This is a key GIS operation: **spatial join**.

### What is a Spatial Join?

Imagine you have:
- A map of all houses (parcels)
- A map of all flood zones (Floodway + 100-year)

A **spatial join** asks: "Which houses fall inside which flood zones?"

Think of it like dropping colored dots (parcels) on a map and seeing which ones land in the colored flood zone. If a parcel overlaps the flood zone at all, they get "joined" together.

### Spatial Relationships
There are different ways to define "which parcels match":
- **Intersects** = Parcel boundary touches or overlaps the flood zone (most common for exposure)
- **Within** = Parcel is completely inside the flood zone
- **Contains** = Flood zone completely contains the parcel

We'll use **'intersects'** — if ANY part of the parcel touches the Floodway or 100-year floodplain, that parcel is exposed.

### CRS (Coordinate Reference System)
Before joining, we make sure both layers use the same **CRS** (Coordinate Reference System).
This tells the computer what part of Earth the data represents and what units are used (meters, feet, degrees, etc.).
If the CRS doesn't match, the spatial join will give wrong results.

In [ ]:
# First, check that both layers have the same CRS (Coordinate Reference System)
print("Checking coordinate systems...")
print(f"Parcels CRS: {parcels.crs}")
print(f"Flood zones CRS: {flood_zones_100yr.crs}")

# If they don't match, reproject to match
if parcels.crs != flood_zones_100yr.crs:
    print(f"\n⚠ CRS mismatch! Reprojecting flood zones to match parcels...")
    flood_zones_100yr = flood_zones_100yr.to_crs(parcels.crs)
    print(f"  Now both use: {parcels.crs}")
else:
    print("✓ Both layers use the same CRS")

### Perform the Spatial Join

Now we'll use the `sjoin()` function (spatial join) to match each parcel to the flood zones it intersects.

In [ ]:
# Spatial join: find which parcels intersect the 100-year floodplain
parcels_joined = gpd.sjoin(
    parcels, flood_zones_100yr,
    how='left', predicate='intersects'
)

print(f"Spatial join completed")
print(f"  Total parcels: {len(parcels):,}")
print(f"  Parcels in joined result: {len(parcels_joined):,}")
print(f"  Parcels with flood zone matches: {parcels_joined['index_right'].notna().sum():,}")

In [ ]:
# Create exposure field: 1 if intersects flood zone, 0 if not
parcels_joined['exposure'] = (parcels_joined['index_right'].notna()).astype(int)

# Flag community asset parcels using parusecode lists
community_assets_all = [str(c) for c in community_asset_1 + community_asset_2]
if 'parusecode' in parcels_joined.columns:
    parcels_joined['is_community_asset'] = parcels_joined['parusecode'].astype(str).isin(community_assets_all).astype(int)
else:
    parcels_joined['is_community_asset'] = 0
    print(f"  Warning: parusecode not found — is_community_asset set to 0")

# Show results
exposure_counts = parcels_joined['exposure'].value_counts()
print("Exposure Results (Floodway + 100-year):")
print(f"  Exposed (1):     {exposure_counts.get(1, 0):>6,} parcels")
print(f"  Not Exposed (0): {exposure_counts.get(0, 0):>6,} parcels")
print(f"  Total:           {len(parcels_joined):>6,} parcels")

total = len(parcels_joined)
exposed_pct = (exposure_counts.get(1, 0) / total * 100)
print(f"\n  {exposed_pct:.1f}% of parcels are exposed to the Floodway + 100-year floodplain")

ca_count = parcels_joined['is_community_asset'].sum()
ca_exposed = ((parcels_joined['is_community_asset'] == 1) & (parcels_joined['exposure'] == 1)).sum()
print(f"\n  Community asset parcels: {ca_count:,} total, {ca_exposed:,} exposed")

## Step 6: Visualize Exposure

Two maps side by side: all parcels by exposure status, and exposed parcels by community asset type.

In [ ]:
# Create a figure with two subplots (side by side) — dark theme
fig, axes = plt.subplots(1, 2, figsize=(16, 8), facecolor='#2b2b2b')

ax1 = axes[0]
ax1.set_facecolor('#2b2b2b')
not_exposed = parcels_joined[parcels_joined['exposure'] == 0]
exposed = parcels_joined[parcels_joined['exposure'] == 1]
if len(not_exposed) > 0:
    not_exposed.plot(ax=ax1, color='#B0BEC5', edgecolor='#333333', linewidth=0.3, alpha=0.6)
if len(exposed) > 0:
    exposed.plot(ax=ax1, color='#E65100', edgecolor='#333333', linewidth=0.3, alpha=0.8)
flood_zones_100yr.plot(ax=ax1, facecolor='none', edgecolor='#2B5797', linewidth=2)
ax1.set_title('All Parcels: Exposure to Floodway + 100-Year Floodplain', fontsize=13, fontweight='bold', color='white')
from matplotlib.patches import Patch
legend_elements_1 = [
    Patch(facecolor='#B0BEC5', edgecolor='#555555', label=f'Not Exposed (0) ({len(not_exposed):,})'),
    Patch(facecolor='#E65100', edgecolor='#555555', label=f'Exposed (1) ({len(exposed):,})'),
    Patch(facecolor='none', edgecolor='#2B5797', linewidth=2, label='Floodway + 100-yr')
]
ax1.legend(handles=legend_elements_1, loc='upper right', fontsize=9, facecolor='#3a3a3a', edgecolor='#555555', labelcolor='white')
ax1.set_axis_off()

ax2 = axes[1]
ax2.set_facecolor('#2b2b2b')
parcels_joined.plot(ax=ax2, color='#3a3a3a', edgecolor='#444444', linewidth=0.2, alpha=0.4)
exposed_parcels = parcels_joined[parcels_joined['exposure'] == 1]

# Asset 1: parcels matching community_asset_1 parusecodes
ca1_codes = [str(c) for c in community_asset_1]
ca1_exposed = exposed_parcels[exposed_parcels['parusecode'].astype(str).isin(ca1_codes)]
if len(ca1_exposed) > 0:
    ca1_exposed.plot(ax=ax2, color='#E65100', edgecolor='#555555', linewidth=0.5, alpha=0.8, label=f'Asset 1 ({len(ca1_exposed):,})')

# Asset 2: parcels matching community_asset_2 parusecodes
ca2_codes = [str(c) for c in community_asset_2]
ca2_exposed = exposed_parcels[exposed_parcels['parusecode'].astype(str).isin(ca2_codes)]
if len(ca2_exposed) > 0:
    ca2_exposed.plot(ax=ax2, color='#E65100', edgecolor='#555555', linewidth=0.5, alpha=0.8, label=f'Asset 2 ({len(ca2_exposed):,})')

# Other exposed parcels
all_ca_codes = ca1_codes + ca2_codes
other_exposed = exposed_parcels[~exposed_parcels['parusecode'].astype(str).isin(all_ca_codes)]
if len(other_exposed) > 0:
    other_exposed.plot(ax=ax2, color='#888888', edgecolor='#555555', linewidth=0.3, alpha=0.6, label=f'Other Exposed ({len(other_exposed):,})')
ax2.set_title('Exposed Parcels by Community Asset', fontsize=13, fontweight='bold', color='white')
ax2.legend(loc='upper right', fontsize=8, facecolor='#3a3a3a', edgecolor='#555555', labelcolor='white')
ax2.set_axis_off()
plt.tight_layout()
plt.show()

## Step 7: Summarize Exposure Statistics

In [ ]:
import contextily as ctx
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

# Reproject to Web Mercator for basemap tiles
parcels_wm = parcels_joined.to_crs(epsg=3857)

# Create figure with map panel + legend panel below
fig = plt.figure(figsize=(10, 12))
gs = gridspec.GridSpec(2, 1, height_ratios=[10, 1.2], hspace=0.02)
ax = fig.add_subplot(gs[0])
ax_legend = fig.add_subplot(gs[1])

# Layer 1 (bottom): All parcels - no fill, thin grey borders
parcels_wm.plot(ax=ax, facecolor='none', edgecolor='#888888', linewidth=0.3)

# Layer 2: Exposure layer - score 0 NOT plotted
# Only plot exposed parcels (score 1)
subset = parcels_wm[parcels_wm['exposure'] == 1]
if len(subset) > 0:
    subset.plot(ax=ax, facecolor='#E65100', edgecolor='none', alpha=0.85)

# Layer 3: Buildings
try:
    buildings_layer = gpd.read_file(CLASS0_GPKG, layer='buildings')
    buildings_wm = buildings_layer.to_crs(epsg=3857)
    buildings_wm.plot(ax=ax, facecolor='#3D3D3D', edgecolor='#2a2a2a', linewidth=0.1, alpha=0.7)
    print(f"✓ Buildings loaded: {len(buildings_layer)} features")
except Exception as e:
    print(f"⚠ Could not load buildings: {e}")

# Layer 4: Flood zones with transparency (all three categories)
try:
    flood_zones_full = gpd.read_file(CLASS0_GPKG, layer='flood_zones')
    flood_wm = flood_zones_full.to_crs(epsg=3857)
    flood_colors = {'Floodway': '#2B5797', '100-year': '#8FABBE', '500-year': '#B4D4E7'}
    for flood_type in ['500-year', '100-year', 'Floodway']:
        flood_subset = flood_wm[flood_wm['flood_category'] == flood_type]
        if len(flood_subset) > 0:
            flood_subset.plot(ax=ax, facecolor=flood_colors.get(flood_type, '#B4D4E7'),
                            edgecolor='none', alpha=0.5)
    print(f"✓ Flood zones loaded: {len(flood_zones_full)} features")
    print(f"  Categories: {flood_zones_full['flood_category'].value_counts().to_dict()}")
except Exception as e:
    print(f"⚠ Could not load flood zones: {e}")

# Add CartoDB Positron (light) basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom='auto')

# Style the map panel
ax.set_axis_off()
ax.set_title('Flood Exposure Assessment', fontsize=14, fontweight='bold', pad=10)

# Build legend in bottom panel
ax_legend.set_axis_off()
legend_elements = [
    Patch(facecolor='none', edgecolor='#cccccc', linewidth=0.5, label='Not Exposed (0)'),
    Patch(facecolor='#E65100', edgecolor='none', label='Exposed (1)'),
    Patch(facecolor='#3D3D3D', edgecolor='#2a2a2a', linewidth=0.1, label='Buildings'),
    Patch(facecolor='#2B5797', edgecolor='none', alpha=0.5, label='Floodway'),
    Patch(facecolor='#8FABBE', edgecolor='none', alpha=0.5, label='100-year Floodplain'),
    Patch(facecolor='#B4D4E7', edgecolor='none', alpha=0.5, label='500-year Floodplain'),
    Patch(facecolor='none', edgecolor='#888888', linewidth=0.5, label='Parcels'),
]
ax_legend.legend(handles=legend_elements, loc='center', ncol=4, fontsize=9,
                frameon=True, facecolor='white', edgecolor='#cccccc',
                handlelength=1.5, handletextpad=0.5, columnspacing=1.5)

# Export
png_path = os.path.join(OUTPUT_DIR, 'exposure_map.png')
plt.savefig(png_path, dpi=150, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f"✓ Map exported to: {png_path}")

## Export Map to PNG

We'll create a high-resolution PNG map showing the exposure assessment with all layers properly styled and ordered.

In [ ]:
# Exposure by Community Asset Group
print("=" * 80)
print("EXPOSURE BY COMMUNITY ASSET GROUP")
print("=" * 80)

ca1_codes = [str(c) for c in community_asset_1]
ca2_codes = [str(c) for c in community_asset_2]
all_ca_codes = ca1_codes + ca2_codes

# Assign each parcel to a group
def assign_group(code):
    code_str = str(code)
    if code_str in ca1_codes:
        return 'Asset 1'
    elif code_str in ca2_codes:
        return 'Asset 2'
    else:
        return 'Other'

parcels_joined['asset_group'] = parcels_joined['parusecode'].astype(str).apply(assign_group)

group_exposure = parcels_joined.groupby('asset_group').agg(
    Total_Parcels=('exposure', 'count'),
    Exposed=('exposure', lambda x: (x == 1).sum()),
    Not_Exposed=('exposure', lambda x: (x == 0).sum())
).reset_index()
group_exposure['Pct_Exposed'] = (group_exposure['Exposed'] / group_exposure['Total_Parcels'] * 100).round(1)

# Display with asset codes for reference
for _, row in group_exposure.iterrows():
    grp = row['asset_group']
    total = int(row['Total_Parcels'])
    exposed = int(row['Exposed'])
    pct = row['Pct_Exposed']
    if grp == 'Asset 1':
        codes_str = f"codes: {ca1_codes}"
    elif grp == 'Asset 2':
        codes_str = f"codes: {ca2_codes}"
    else:
        codes_str = "all other parusecodes"
    print(f"\n  {grp} ({codes_str})")
    print(f"    {total:,} total parcels | {exposed:,} exposed ({pct:.1f}%)")

# Grand total
total_all = len(parcels_joined)
exposed_all = (parcels_joined['exposure'] == 1).sum()
print(f"\n  {'─' * 60}")
print(f"  TOTAL: {total_all:,} parcels | {exposed_all:,} exposed ({exposed_all/total_all*100:.1f}%)")

In [ ]:
# Value-based exposure by Asset Group
print("=" * 80)
print("EXPOSURE BY PROPERTY VALUE (by Asset Group)")
print("=" * 80)
value_columns = ['parval', 'improvval', 'landval']
available_values = [col for col in value_columns if col in parcels_joined.columns]
if available_values:
    value_col = available_values[0]
    print(f"Using column: {value_col}\n")

    # Deduplicate parcels_joined to one row per parcel
    # The spatial join creates duplicate rows when a parcel intersects multiple
    # flood zone features. Rule: if ANY portion of a parcel is in the flood zone,
    # that parcel is exposed (exposure = 1). Only parcels with NO intersection
    # are not exposed (exposure = 0).
    #
    # We use groupby on the original parcel index and take max(exposure) so that
    # any parcel that matched at least one flood zone feature keeps exposure = 1.
    agg_dict = {col: 'first' for col in parcels_joined.columns if col != 'geometry'}
    agg_dict['exposure'] = 'max'  # ANY intersection → exposed
    agg_dict['geometry'] = 'first'
    # Remove join artifacts from aggregation
    for drop_col in ['index_right', 'flood_category', 'FLD_ZONE', 'ZONE_SUBTY', 'flood_zone']:
        agg_dict.pop(drop_col, None)

    parcels_dedup = parcels_joined.groupby(parcels_joined.index).agg(agg_dict)
    print(f"  Deduplicated: {len(parcels_joined):,} joined rows → {len(parcels_dedup):,} unique parcels")
    print(f"  Original parcels: {len(parcels):,}")

    # Grand totals for percentage calculations
    grand_total_parcels = len(parcels_dedup)
    grand_total_value = parcels_dedup[value_col].sum()

    # Build summary table
    rows = []
    for grp in ['Asset 1', 'Asset 2', 'Other']:
        subset = parcels_dedup[parcels_dedup['asset_group'] == grp]
        if len(subset) == 0:
            continue
        n_parcels = len(subset)
        n_exposed = (subset['exposure'] == 1).sum()
        total_val = subset[value_col].sum()
        exposed_val = subset[subset['exposure'] == 1][value_col].sum()

        rows.append({
            'Asset Group': grp,
            'Parcels': n_parcels,
            'Pct of Total Parcels': round(n_parcels / grand_total_parcels * 100, 1),
            'Total Value': total_val,
            'Pct of Total Value': round(total_val / grand_total_value * 100, 1) if grand_total_value > 0 else 0,
            'Exposed Parcels': n_exposed,
            'Pct Exposed Parcels': round(n_exposed / n_parcels * 100, 1) if n_parcels > 0 else 0,
            'Exposed Value': exposed_val,
            'Pct Exposed Value': round(exposed_val / total_val * 100, 1) if total_val > 0 else 0
        })

    summary_df = pd.DataFrame(rows)

    # Display
    for _, row in summary_df.iterrows():
        print(f"  {row['Asset Group']}:")
        print(f"    Parcels:       {int(row['Parcels']):>8,}  ({row['Pct of Total Parcels']:>5.1f}% of total)")
        print(f"    Total Value:   ${row['Total Value']:>15,.0f}  ({row['Pct of Total Value']:>5.1f}% of total)")
        print(f"    Exposed:       {int(row['Exposed Parcels']):>8,} parcels ({row['Pct Exposed Parcels']:>5.1f}%)  |  ${row['Exposed Value']:>15,.0f}  ({row['Pct Exposed Value']:>5.1f}% of group value)")
        print()

    # Grand total line
    total_exposed_val = parcels_dedup[parcels_dedup['exposure'] == 1][value_col].sum()
    total_exposed_n = (parcels_dedup['exposure'] == 1).sum()
    print(f"  {'─' * 70}")
    print(f"  TOTAL: {grand_total_parcels:,} parcels  |  ${grand_total_value:>15,.0f}")
    print(f"  EXPOSED: {total_exposed_n:,} parcels ({total_exposed_n/grand_total_parcels*100:.1f}%)  |  ${total_exposed_val:>15,.0f}  ({total_exposed_val/grand_total_value*100:.1f}% of total value)")

    # Store summary_df for saving in the GeoPackage save cell (Step 8)
    exposure_value_summary = summary_df.copy()
    exposure_value_summary['Total Value'] = exposure_value_summary['Total Value'].astype(float)
    exposure_value_summary['Exposed Value'] = exposure_value_summary['Exposed Value'].astype(float)
    print(f"\n  (Summary table will be saved to GeoPackage in Step 8)")
else:
    exposure_value_summary = None
    print("Note: No property value columns found in the data.")

## Step 8: Save Results Back to GeoPackage

We save exposure scores and community asset flags to the GeoPackage so future notebooks can use them.

## Appendix A: Doing This in QGIS

### Step 1: Open Data in QGIS
1. Open QGIS and create a new project
2. Use **Layer > Add Layer > Add Vector Layer**
3. Browse to `vulnerability_risk_data.gpkg` and select both layers:
   - `parcels`
   - `flood_zones`

### Step 2: Filter Flood Zones to Floodway + 100-Year
1. Right-click the `flood_zones` layer > **Filter...**
2. Enter the expression: `"flood_category" IN ('100-year', 'Floodway')`
3. Click **OK** — the layer now shows only Floodway and 100-year floodplain areas

### Step 3: Select Exposed Parcels (Select by Location)
1. Go to **Vector > Research Tools > Select by Location**
2. Set parameters:
   - Select features from: `parcels`
   - Where the features: **intersect**
   - By comparing to: `flood_zones` (filtered to Floodway + 100-year)
3. Click **Run** — exposed parcels are now selected (highlighted)

### Step 4: Add Exposure Field
1. Open the `parcels` attribute table (right-click > Open Attribute Table)
2. Toggle **Edit Mode** (pencil icon)
3. Click **New Field** (Ctrl+W):
   - Name: `exposure`
   - Type: Integer
4. Open **Field Calculator** (abacus icon):
   - Check "Update existing field" and select `exposure`
   - Expression: `if(is_selected(), 1, 0)`
5. Click **OK**, then save edits and exit edit mode

### Step 5: Visualize Exposure
1. Right-click `parcels` > **Symbology**
2. Change to **Categorized**
3. Column: `exposure`
4. Click **Classify**
5. Set colors: 0 = green, 1 = red
6. Add `flood_zones` layer with blue outline (no fill)

### Step 6: Get Statistics
1. Go to **Vector > Analysis Tools > Basic Statistics for Fields**
2. Input layer: `parcels`
3. Field: `exposure`
4. This gives count, sum (number exposed), mean (proportion exposed)

In [ ]:
# Save results to a NEW GeoPackage for Class 1
# This does NOT overwrite the original vulnerability_risk_data.gpkg from Class 0.

import fiona
import sqlite3

parcels_output = parcels_joined.copy()
parcels_output = parcels_output.drop(columns=['index_right'], errors='ignore')
parcels_output['exposure'] = parcels_output['exposure'].astype(int)

output_path = os.path.join(DATA_DIR, 'class_1_exposure.gpkg')
try:
    # Write spatial layers first (creates/overwrites the GeoPackage)
    parcels_output.to_file(output_path, layer='parcels', driver='GPKG')
    print(f"✓ Saved parcels layer ({len(parcels_output):,} features)")

    # Save the filtered 100-year flood zones used for exposure analysis
    flood_zones_100yr.to_file(output_path, layer='flood_zones_100yr', driver='GPKG', mode='a')
    print(f"✓ Saved flood_zones_100yr layer ({len(flood_zones_100yr):,} features)")

    # Write the summary table (non-spatial, via sqlite3)
    if exposure_value_summary is not None:
        conn = sqlite3.connect(output_path)
        exposure_value_summary.to_sql('summary', conn, if_exists='replace', index=False)
        conn.close()
        print(f"✓ Saved summary table ({len(exposure_value_summary)} rows)")

    # Carry forward ALL base layers from Class 0 (flood_zones with 500-year, buildings, study_area)
    if os.path.exists(CLASS0_GPKG):
        output_layers = fiona.listlayers(output_path)
        class0_layers = fiona.listlayers(CLASS0_GPKG)
        for layer_name in class0_layers:
            if layer_name not in output_layers:
                try:
                    layer_data = gpd.read_file(CLASS0_GPKG, layer=layer_name)
                    layer_data.to_file(output_path, layer=layer_name, driver='GPKG', mode='a')
                    print(f"✓ Carried forward from Class 0: {layer_name} ({len(layer_data)} features)")
                except Exception as e:
                    print(f"  Note: Could not copy base layer '{layer_name}': {e}")
    else:
        print(f"⚠ CLASS0_GPKG not found at: {CLASS0_GPKG}")

    # Final summary
    final_layers = fiona.listlayers(output_path)
    print(f"\n✓ All data saved to: {output_path}")
    print(f"  File size: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB")
    print(f"  Layers: {final_layers}")
    print(f"\n  Class 2 will load from this file.")
except Exception as e:
    print(f"Error saving: {e}")

## Appendix B: Doing This in ArcGIS Pro

### Step 1: Open Data in ArcGIS Pro
1. Create new project
2. Use **Insert > Add Data** to load from GeoPackage:
   - vulnerability_risk_data.gpkg
   - Add parcels and flood_zones layers

### Step 2: Filter Flood Zones
1. Right-click flood_zones layer > Attribute Table
2. Click **Filter** (funnel icon)
3. Set filter: `flood_category IN ('100-year', 'Floodway')`

### Step 3: Select Exposed Parcels Using Select by Location
1. Select the parcels layer
2. Go to **Map** tab > Select by Location
3. Set parameters:
   - Input layer: parcels
   - Relationship: Intersect
   - Selecting features: flood_zones (Floodway + 100-year)
4. Click Run
5. This selects all parcels that intersect the floodplain

### Step 4: Add Exposure Field Using Field Calculator
1. Right-click parcels layer > Attribute Table
2. Click **Add Field**
   - Name: exposure
   - Type: Integer
3. Right-click the new field > Calculate Field
4. In the **Field Calculator**:
   - Expression type: Arcade (or Python)
   - Use this Arcade expression:
     ```
     if(Intersects($feature.geometry, FeatureSet(Text("{flood_zones_layer}", "geometry"))), 1, 0)
     ```
   
   Or if you used Select by Location, simpler approach:
     ```
     if($feature.OBJECTID in [list of selected OBJECTID values], 1, 0)
     ```

### Step 5: Add Asset Type Field
1. Add Field: asset_type (Text, 50)
2. Use Field Calculator with Arcade:
   ```
   var code = $feature.parusecode;
   if(Find(code, "R") == 0) { return "Residential"; }
   else if(Find(code, "C") == 0) { return "Commercial"; }
   else if(Find(code, "I") == 0) { return "Industrial"; }
   else if(Find(code, "O") == 0 || Find(code, "10") == 0) { return "Open Space/Vacant"; }
   else { return "Other"; }
   ```

### Step 6: Create Maps
1. Change symbology of parcels:
   - Right-click parcels > Symbology
   - Choose: Single Symbol or Unique Values
   - Symbol by exposure field
   - Assign red for 1 (exposed), green for 0

2. Add flood_zones layer with blue outline, no fill

3. Export map:
   - Share tab > Export as Image or PDF

### Step 7: Get Statistics
Use Geoprocessing tools:
- **Summarize** (Spatial Statistics toolbox) to create a summary table
- Or right-click layer > Attribute Table > Summarize

## Summary

Congratulations! You've completed **Class 1: Understanding Exposure**.

### What You Did
1. ✓ Loaded parcel and flood zone data from a GeoPackage
2. ✓ Explored three land use classification fields and chose your community asset field
3. ✓ Filtered to the **Floodway + 100-year floodplain** (the regulatory Special Flood Hazard Area)
4. ✓ Used a **spatial join** to identify which parcels are exposed
5. ✓ Created a **binary exposure field** (1 = exposed, 0 = not exposed)
6. ✓ Visualized results on dark-themed maps
7. ✓ Calculated exposure statistics by land use and property value
8. ✓ Saved results back to the GeoPackage for use in future classes

### Key Concepts Learned
- **Exposure** = Is the asset in the path of the hazard?
- **Floodway** = The river channel and adjacent areas that must remain clear for flood flow (most dangerous)
- **100-year floodplain** = 1% annual probability (NOT once every 100 years!)
- **Binary scoring** = Simple yes/no answers to exposure
- **Spatial join** = Matching features based on geography
- **Coordinate Reference System (CRS)** = How the computer knows where on Earth your data is
- **Asset field selection** = Choosing the best classification field for your analysis

### Next Class
In **Class 2**, we'll look at **Potential Impact** — if a parcel gets flooded, how much damage would it suffer? This depends on the building type, construction quality, and contents.

Then in **Class 3**, we'll assess **Adaptive Capacity** — how well can the community recover?

---

**Questions?** Review this notebook, ask your instructor, or consult the QGIS/ArcGIS sections in the appendices to see how this works in desktop GIS software.